# Creating Opportunity Score Index

In [1]:
import pandas as pd
import numpy as np


df = pd.read_csv("data/processed/engine_latest_data.csv")

In [2]:
required_columns = [
    "state",
    "industry",
    "nsva",
    "industry_share",
    "industry_cagr_3y",
    "startup_cagr_3y",
    "underpenetration_score"
]

missing = [
    col for col in required_columns
    if col not in df.columns
]

if missing:
    raise ValueError(
        f"Missing columns: {missing}"
    )


# 3. CLEAN DATA

df = df.replace(
    [np.inf, -np.inf],
    np.nan
)

# Numeric columns
score_variables = [
    "nsva",
    "industry_share",
    "industry_cagr_3y",
    "startup_cagr_3y",
    "underpenetration_score"
]

for col in score_variables:
    df[col] = pd.to_numeric(
        df[col],
        errors="coerce"
    )

In [3]:

# ============================================================
# 4. CONVERT VARIABLES TO 0–100 SCORES
# ============================================================

# Percentile ranking is used rather than raw min-max scaling.

# This makes the engine more robust to extreme values.

# IMPORTANT:
# Ranking is performed WITHIN EACH INDUSTRY.
# Therefore we answer:
#
# "Which states perform best for THIS industry?"
# ============================================================

df["economic_strength_score"] = (
    df
    .groupby("industry")["nsva"]
    .rank(pct=True) * 100
)

df["industry_growth_score"] = (
    df
    .groupby("industry")["industry_cagr_3y"]
    .rank(pct=True) * 100
)

df["startup_momentum_score"] = (
    df
    .groupby("industry")["startup_cagr_3y"]
    .rank(pct=True) * 100
)

df["industry_presence_score"] = (
    df
    .groupby("industry")["industry_share"]
    .rank(pct=True) * 100
)

# Already calculated as a 0–100 score
df["underpenetration_score"] = (
    df["underpenetration_score"]
    .clip(0, 100)
)

In [4]:

# ============================================================
# 5. DEFINE STRATEGIES
# ============================================================
#
# Three business strategies:
#
# 1. Balanced
# 2. Growth
# 3. Market Entry
#
# We deliberately keep the weights transparent.
# ============================================================

strategies = {

    "Balanced": {
        "economic_strength_score": 0.20,
        "industry_growth_score": 0.20,
        "startup_momentum_score": 0.20,
        "underpenetration_score": 0.20,
        "industry_presence_score": 0.20
    },

    "Growth": {
        "economic_strength_score": 0.10,
        "industry_growth_score": 0.35,
        "startup_momentum_score": 0.25,
        "underpenetration_score": 0.20,
        "industry_presence_score": 0.10
    },

    "Market Entry": {
        "economic_strength_score": 0.30,
        "industry_growth_score": 0.20,
        "startup_momentum_score": 0.10,
        "underpenetration_score": 0.30,
        "industry_presence_score": 0.10
    }
}

In [5]:
# 6. CALCULATE OPPORTUNITY SCORES

for strategy_name, weights in strategies.items():

    df[f"{strategy_name.lower()}_opportunity_score"] = (

        df["economic_strength_score"]
        * weights["economic_strength_score"]

        +

        df["industry_growth_score"]
        * weights["industry_growth_score"]

        +

        df["startup_momentum_score"]
        * weights["startup_momentum_score"]

        +

        df["underpenetration_score"]
        * weights["underpenetration_score"]

        +

        df["industry_presence_score"]
        * weights["industry_presence_score"]
    )


# 7. CREATE RANKINGS

for strategy_name in strategies.keys():

    score_column = (
        f"{strategy_name.lower()}_opportunity_score"
    )

    rank_column = (
        f"{strategy_name.lower()}_rank"
    )

    df[rank_column] = (
        df
        .groupby("industry")[score_column]
        .rank(
            ascending=False,
            method="min"
        )
    )


# 8. CREATE PRIMARY / BALANCED RANK

df["opportunity_score"] = (
    df["balanced_opportunity_score"]
)

df["opportunity_rank"] = (
    df["balanced_rank"]
)


# 9. CREATE OPPORTUNITY TIER

def assign_tier(rank, total):

    high_cutoff = int(np.ceil(total * 0.25))
    low_cutoff = int(np.ceil(total * 0.75))

    if rank <= high_cutoff:
        return "High Opportunity"

    elif rank <= low_cutoff:
        return "Moderate Opportunity"

    else:
        return "Lower Opportunity"


df["opportunity_tier"] = df.apply(
    lambda row: assign_tier(
        row["opportunity_rank"],
        df[
            df["industry"] == row["industry"]
        ]["state"].nunique()
    ),
    axis=1
)

In [6]:
# 10. CREATE "WHY THIS STATE?" COMPONENTS

component_columns = {
    "Economic Strength": "economic_strength_score",
    "Industry Growth": "industry_growth_score",
    "Startup Momentum": "startup_momentum_score",
    "Underpenetration": "underpenetration_score",
    "Industry Presence": "industry_presence_score"
}


def generate_reason(row):

    scores = {
        name: row[column]
        for name, column in component_columns.items()
        if pd.notna(row[column])
    }

    ranked = sorted(
        scores.items(),
        key=lambda x: x[1],
        reverse=True
    )

    top_two = ranked[:2]

    return ", ".join(
        [item[0] for item in top_two]
    )


df["key_drivers"] = df.apply(
    generate_reason,
    axis=1
)


In [7]:
# 11. FINAL ENGINE DATASET

engine_output_columns = [

    "year",
    "state",
    "industry",

    # Raw business metrics
    "nsva",
    "industry_startups",
    "industry_share",
    "industry_cagr_3y",
    "startup_cagr_3y",

    # Engine scores
    "economic_strength_score",
    "industry_growth_score",
    "startup_momentum_score",
    "underpenetration_score",
    "industry_presence_score",

    # Opportunity scores
    "balanced_opportunity_score",
    "growth_opportunity_score",
    "market entry_opportunity_score",

    # Rankings
    "balanced_rank",
    "growth_rank",
    "market entry_rank",

    # Interpretation
    "opportunity_tier",
    "key_drivers"
]


engine_output = df[
    engine_output_columns
].copy()

In [8]:

# 12. ROUND SCORES

score_columns = [
    "economic_strength_score",
    "industry_growth_score",
    "startup_momentum_score",
    "underpenetration_score",
    "industry_presence_score",
    "balanced_opportunity_score",
    "growth_opportunity_score",
    "market entry_opportunity_score"
]

engine_output[score_columns] = (
    engine_output[score_columns]
    .round(2)
)


# 13. SAVE ENGINE

engine_output.to_csv(
    "data/processed/opportunity_engine.csv",
    index=False
)

print("\nOpportunity Engine saved:")
print("opportunity_engine.csv")


Opportunity Engine saved:
opportunity_engine.csv


In [9]:
# 14. EXAMPLE — TOP MARKETS

selected_industry = "Finance Technology"

industry_results = (
    engine_output[
        engine_output["industry"]
        == selected_industry
    ]
    .sort_values(
        "balanced_opportunity_score",
        ascending=False
    )
)

print("\n" + "=" * 65)
print(
    f"TOP OPPORTUNITIES — {selected_industry}"
)
print("=" * 65)

print(
    industry_results[
        [
            "state",
            "balanced_opportunity_score",
            "growth_opportunity_score",
            "market entry_opportunity_score",
            "balanced_rank",
            "opportunity_tier",
            "key_drivers"
        ]
    ]
    .head(10)
    .to_string(index=False)
)


TOP OPPORTUNITIES — Finance Technology
         state  balanced_opportunity_score  growth_opportunity_score  market entry_opportunity_score  balanced_rank     opportunity_tier                          key_drivers
    Tamil Nadu                       68.55                     61.88                           70.32            1.0     High Opportunity Industry Presence, Economic Strength
     Rajasthan                       64.16                     65.41                           59.57            2.0     High Opportunity   Industry Growth, Industry Presence
         Bihar                       58.65                     68.65                           48.81            3.0     High Opportunity    Industry Growth, Startup Momentum
     Karnataka                       57.04                     46.21                           57.23            4.0 Moderate Opportunity Industry Presence, Economic Strength
       Gujarat                       56.60                     56.18                      

In [10]:
# 15. TOP 10 FOR EACH INDUSTRY

top_markets = (
    engine_output
    .sort_values(
        [
            "industry",
            "balanced_opportunity_score"
        ],
        ascending=[True, False]
    )
    .groupby("industry")
    .head(10)
)

top_markets.to_csv(
    "data/processed/top_market_opportunities.csv",
    index=False
)

print("\nTop market rankings saved:")
print("top_market_opportunities.csv")



Top market rankings saved:
top_market_opportunities.csv
